# Stg Layer — video_classification (LLM, Incremental)

Phân loại mỗi video (`${silver}.stg_videos`, latest snapshot) từ **title / description / tags / category** bằng `ai_query` → Silver `${silver}.stg_video_classification` (1 row / video). Mục đích: các bước tốn API ở sau (Gemini transcript, `ai_query` trích quán, rồi Places) chỉ xử lý video có thể là review quán.

**Nhãn (`video_type`):**
- `restaurant_review`: video giới thiệu / review / đi ăn ở quán, nhà hàng, tiệm, xe đẩy cụ thể (kể cả list "top N quán").
- `food_other`: về đồ ăn uống nhưng không gắn với quán cụ thể (nấu ăn, công thức, review sản phẩm đóng gói, mukbang tại nhà, kiến thức món ăn).
- `not_food`: không phải đồ ăn uống.

**`is_food_review`** (cờ cho serving, tính ở đây để mọi task dùng chung 1 luật): `false` chỉ khi `video_type = 'not_food'` và `confidence ≥ MIN_CONFIDENCE`. Ưu tiên không bỏ sót: bỏ sót 1 review là mất quán, transcribe thừa chỉ tốn quota. Video chưa phân loại được coi là `true`. Gold copy cờ này vào `dim_video` (SCD 1). Video **không bị xóa**, chỉ được đánh dấu.

**Incremental:** `source_checksum = md5(title || description || tags || category_id || PROMPT_VERSION)`; chỉ gọi LLM cho video mới hoặc đổi checksum. `PROMPT_VERSION` = md5 của model + prompt, nên sửa prompt thì mọi video được phân loại lại 1 lần. Đổi `MIN_CONFIDENCE` không cần gọi lại LLM (cờ được tính lại cho mọi row).

**So sánh (cell cuối):** đối chiếu với `stg_restaurant_extraction_log` (video nào đã trích ra quán): video có quán mà bị `is_food_review = false` là bỏ sót.

In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

VIDEOS_TABLE = f"{SILVER}.stg_videos"
CLASS_TABLE = f"{SILVER}.stg_video_classification"
EXTRACTION_LOG = f"{SILVER}.stg_restaurant_extraction_log"  # only for the comparison cell
TRANSCRIPTS_TABLE = f"{SILVER}.stg_video_transcripts"        # only for the comparison cell
MODEL = "databricks-qwen3-next-80b-a3b-instruct"  # same model as restaurant_mentions
MAX_DESCRIPTION_CHARS = 3000
MIN_CONFIDENCE = 0.7  # not_food below this stays is_food_review = true (recall first)

PROMPT = (
    "Classify a Vietnamese YouTube video from its title, description, tags and category. "
    "Labels: "
    "restaurant_review = the video visits, reviews or recommends one or more specific places to eat or drink "
    "(quán ăn, nhà hàng, quán cà phê, tiệm, xe đẩy, buffet, food tour, a 'top N quán' list); "
    "food_other = about food or drink but not tied to a specific place (cooking, recipes, packaged product reviews, "
    "eating at home, food facts, food challenges); "
    "not_food = anything else (music, comedy, travel without food places, news, tips, ads for non-food products). "
    "A title that is mostly a street address (e.g. '126 Âu Cơ, Phường 9, Tân Bình') is usually a short showing that eating place: "
    "label it restaurant_review unless the rest says otherwise. "
    "confidence = your probability (0 to 1) that the label is right. reason = at most 12 words."
)
RESPONSE_FORMAT = """{"type":"json_schema","json_schema":{"name":"video_class","schema":{"type":"object","properties":{"video_type":{"type":"string","enum":["restaurant_review","food_other","not_food"]},"confidence":{"type":"number"},"reason":{"type":"string"}},"required":["video_type","confidence","reason"]},"strict":true}}"""
RESULT_SCHEMA = "STRUCT<video_type: STRING, confidence: DOUBLE, reason: STRING>"
# Prompt / model version: part of source_checksum, so editing the prompt or model re-classifies every video once
import hashlib
PROMPT_VERSION = hashlib.md5((MODEL + PROMPT + RESPONSE_FORMAT).encode()).hexdigest()[:8]

print(f"Videos source: {VIDEOS_TABLE}")
print(f"Target: {CLASS_TABLE} · model {MODEL} · not_food skipped when confidence ≥ {MIN_CONFIDENCE}")

In [ ]:
# --- Create table if not exists ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CLASS_TABLE} (
        video_id         STRING,
        video_type       STRING,
        confidence       DOUBLE,
        reason           STRING,
        is_food_review   BOOLEAN,
        source_checksum  STRING,
        raw_response     STRING,
        model_name       STRING,
        classified_at    TIMESTAMP
    )
    USING DELTA
    COMMENT 'LLM video type per video (1 row per video). is_food_review = false only for confident not_food: serving tasks skip those.'
""")
print("Table ready ✅")

In [ ]:
# --- Videos to (re-)classify: new, or title / description / tags / category changed ---
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW videos_to_classify AS
    WITH latest AS (
        SELECT v.video_id, v.title, v.description, v.tags, v.category_id,
               md5(concat_ws('||', coalesce(v.title, ''), coalesce(v.description, ''),
                             coalesce(array_join(v.tags, ','), ''), coalesce(v.category_id, ''), '{PROMPT_VERSION}')) AS source_checksum,
               ROW_NUMBER() OVER (PARTITION BY v.video_id ORDER BY v.snapshot_date DESC, v.updated_at DESC) AS rn
        FROM {VIDEOS_TABLE} v
        WHERE v.video_id IS NOT NULL
    )
    SELECT l.video_id, l.title, l.description, l.tags, l.category_id, l.source_checksum
    FROM latest l
    LEFT JOIN {CLASS_TABLE} c ON c.video_id = l.video_id
    WHERE l.rn = 1 AND (c.video_id IS NULL OR c.source_checksum <> l.source_checksum)
""")
todo_count = spark.table("videos_to_classify").count()
print(f"Videos to classify this run: {todo_count}")

In [ ]:
# --- Classify with ai_query and MERGE (LLM is called once per new / changed video) ---
CATEGORY_SQL = """CASE category_id WHEN '1' THEN 'Film & Animation' WHEN '2' THEN 'Autos & Vehicles' WHEN '10' THEN 'Music'
    WHEN '15' THEN 'Pets & Animals' WHEN '17' THEN 'Sports' WHEN '19' THEN 'Travel & Events' WHEN '20' THEN 'Gaming'
    WHEN '22' THEN 'People & Blogs' WHEN '23' THEN 'Comedy' WHEN '24' THEN 'Entertainment' WHEN '25' THEN 'News & Politics'
    WHEN '26' THEN 'Howto & Style' WHEN '27' THEN 'Education' WHEN '28' THEN 'Science & Technology' ELSE '' END"""
if todo_count > 0:
    prompt_sql = PROMPT.replace("'", "''")
    spark.sql(f"""
        MERGE INTO {CLASS_TABLE} AS t
        USING (
            SELECT video_id, r.video_type, r.confidence, r.reason,
                   NOT (coalesce(r.video_type, '') = 'not_food' AND coalesce(r.confidence, 0) >= {MIN_CONFIDENCE}) AS is_food_review,
                   source_checksum, raw_response, '{MODEL}' AS model_name, current_timestamp() AS classified_at
            FROM (
                SELECT *, from_json(raw_response, '{RESULT_SCHEMA}') AS r
                FROM (
                    SELECT video_id, source_checksum,
                           ai_query('{MODEL}',
                                    concat('{prompt_sql}', '\\n\\nTITLE: ', coalesce(title, ''),
                                           '\\n\\nCATEGORY: ', {CATEGORY_SQL},
                                           '\\n\\nTAGS: ', coalesce(array_join(tags, ', '), ''),
                                           '\\n\\nDESCRIPTION: ', left(coalesce(description, ''), {MAX_DESCRIPTION_CHARS})),
                                    responseFormat => '{RESPONSE_FORMAT}') AS raw_response
                    FROM videos_to_classify
                )
            )
        ) AS s
        ON t.video_id = s.video_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"✅ Classified {todo_count} video(s) with {MODEL}")
else:
    print("Nothing new to classify ✅")

# --- Re-apply the threshold to every row (MIN_CONFIDENCE may have changed; no LLM call) ---
spark.sql(f"""
    UPDATE {CLASS_TABLE}
    SET is_food_review = NOT (coalesce(video_type, '') = 'not_food' AND coalesce(confidence, 0) >= {MIN_CONFIDENCE})
    WHERE is_food_review IS DISTINCT FROM NOT (coalesce(video_type, '') = 'not_food' AND coalesce(confidence, 0) >= {MIN_CONFIDENCE})
""")

In [ ]:
# --- Verify ---
stats = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT video_id) AS n_grain, COUNT_IF(video_type IS NULL) AS unparsed,
           COUNT_IF(video_type = 'restaurant_review') AS restaurant_review, COUNT_IF(video_type = 'food_other') AS food_other,
           COUNT_IF(video_type = 'not_food') AS not_food, COUNT_IF(NOT is_food_review) AS skipped
    FROM {CLASS_TABLE}
""").collect()[0]
print(f"📊 {CLASS_TABLE}: {stats['n']} videos · restaurant_review {stats['restaurant_review']} · "
      f"food_other {stats['food_other']} · not_food {stats['not_food']} · unparsed {stats['unparsed']}")
print(f"📊 is_food_review = false (skipped by serving): {stats['skipped']}")
print(f"📊 Grain check (video_id): {stats['n']} rows vs {stats['n_grain']} distinct "
      + ("✅" if stats["n"] == stats["n_grain"] else "❌ duplicates"))

## So sánh với kết quả trích quán

`has_restaurant` = `stg_restaurant_extraction_log.restaurant_count > 0` (ai_query đã tìm được quán từ title / description / transcript). Video có quán mà `is_food_review = false` là **bỏ sót** (mục tiêu: 0). Bảng cuối cho biết bao nhiêu lần gọi Gemini được tiết kiệm (video ≤ 20 phút chưa có transcript).

In [ ]:
# --- Compare with the restaurant extraction (only when its log exists) ---
import json

if spark.catalog.tableExists(EXTRACTION_LOG):
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW class_vs_extraction AS
        WITH latest AS (
            SELECT video_id, title,
                   ROW_NUMBER() OVER (PARTITION BY video_id ORDER BY snapshot_date DESC, updated_at DESC) AS rn
            FROM {VIDEOS_TABLE}
        )
        SELECT c.video_id, l.title, c.video_type, c.confidence, c.reason, c.is_food_review,
               coalesce(g.restaurant_count, 0) > 0 AS has_restaurant, g.video_id IS NOT NULL AS extracted
        FROM {CLASS_TABLE} c
        JOIN latest l ON l.video_id = c.video_id AND l.rn = 1
        LEFT JOIN {EXTRACTION_LOG} g ON g.video_id = c.video_id
    """)
    matrix = spark.sql("""
        SELECT video_type, is_food_review, COUNT_IF(has_restaurant) AS with_restaurant,
               COUNT_IF(extracted AND NOT has_restaurant) AS without_restaurant, COUNT_IF(NOT extracted) AS not_extracted,
               COUNT(*) AS videos
        FROM class_vs_extraction GROUP BY ALL ORDER BY video_type, is_food_review
    """)
    display(matrix)
    missed = spark.sql("""
        SELECT video_id, title, video_type, confidence, reason FROM class_vs_extraction
        WHERE has_restaurant AND NOT is_food_review ORDER BY confidence DESC
    """)
    print(f"❗ Videos with a restaurant but is_food_review = false (missed): {missed.count()}")
    display(missed)
    # Food videos where no restaurant was found (food_other / restaurant_review), to eyeball the labels
    display(spark.sql("""
        SELECT video_type, confidence, has_restaurant, title, reason FROM class_vs_extraction
        ORDER BY video_type, has_restaurant, confidence DESC
    """))
    summary = {r["video_type"] + ("" if r["is_food_review"] else " (skipped)"):
               {"videos": r["videos"], "with_restaurant": r["with_restaurant"]} for r in matrix.collect()}
    # Titles in the run output too, so a chat session can review the labels via the Jobs API (runs/get-output)
    summary["missed"] = [f"{r['title']} | {r['video_type']} {r['confidence']} | {r['reason']}" for r in missed.collect()]
    summary["samples"] = {t: [f"{r['title'][:70]} | {r['confidence']} | rest={r['has_restaurant']}" for r in spark.sql(f"""
        SELECT title, confidence, has_restaurant FROM class_vs_extraction WHERE video_type = '{t}'
        ORDER BY has_restaurant DESC, confidence, video_id LIMIT 40""").collect()]
        for t in ("restaurant_review", "food_other", "not_food")}
else:
    summary = {}
    print(f"{EXTRACTION_LOG} does not exist yet: nothing to compare")

# --- Gemini calls saved: videos ≤ 20 min without a transcript that serving now skips ---
if spark.catalog.tableExists(TRANSCRIPTS_TABLE) and spark.catalog.tableExists(f"{GOLD}.dim_video"):
    saved = spark.sql(f"""
        SELECT COUNT(*) AS waiting, COUNT_IF(NOT coalesce(c.is_food_review, true)) AS skipped
        FROM {GOLD}.dim_video v
        LEFT ANTI JOIN {TRANSCRIPTS_TABLE} t ON t.video_id = v.video_id
        LEFT JOIN {CLASS_TABLE} c ON c.video_id = v.video_id
        WHERE v.duration_seconds > 0 AND v.duration_seconds <= 20 * 60
    """).collect()[0]
    print(f"📊 Videos waiting for a transcript: {saved['waiting']} · skipped by is_food_review: {saved['skipped']}")
    summary["transcripts_waiting"], summary["transcripts_skipped"] = saved["waiting"], saved["skipped"]

print("\n--- stg_video_classification ETL complete ---")
dbutils.notebook.exit(json.dumps(summary, ensure_ascii=False))